# Example 11.3.2 / 3: Control Substitution with substout

Companion notebook for **11.3.2 Substitution and Expression Expansion** in the AMPL lab Documentation, section "Control Substitution with substout". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Control Substitution with substout

The same relation can be written as an ordinary constraint. With `substout 1`, AMPL scans
eligible equalities of the form `variable = expression` and communicates them as defined
variables. The variable must be unrestricted (no bounds or integrality) and must not already have
appeared in an earlier eligible defining constraint, so declaration order can affect which
substitutions occur.

The cost written as an explicit constraint `CostDef` on an unrestricted `Cost`. With the default
`substout 0` the solver sees two variables and one constraint; after
`ampl.option["substout"] = 1` it sees one variable and none, while the optimum is unchanged:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var Ship >= 2, <= 10, := 3;  # shipment, bounded and started at 3
var Cost;                    # unrestricted unit cost

minimize Total: Cost * Ship;  # total cost, unit cost times shipment

subject to CostDef: Cost = (10 + 4 * Ship) / (1 + Ship);  # defining constraint substout can remove

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for value in (0, 1):
    ampl.option["substout"] = value
    ampl.solve(solver="ipopt", verbose=False)
    print("substout", value, ampl.get_value("_snvars"), ampl.get_value("_sncons"),
          round(ampl.obj["Total"].value(), 6))

substout 0 2 1 12.0
substout 1 1 0 12.0


### Pure AMPL: model.run

Pure AMPL, for `substout 1`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver ipopt;
option substout 1;
solve > ipopt.log;
display _snvars, _sncons, Total;

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
_snvars = 1
_sncons = 0
Total = 12

